In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/amazon-delivery-dataset/amazon_delivery.csv


In [2]:
import warnings

warnings.simplefilter('ignore')

## Data Overview

In [3]:
df = pd.read_csv('/kaggle/input/amazon-delivery-dataset/amazon_delivery.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         43739 non-null  object 
 1   Agent_Age        43739 non-null  int64  
 2   Agent_Rating     43685 non-null  float64
 3   Store_Latitude   43739 non-null  float64
 4   Store_Longitude  43739 non-null  float64
 5   Drop_Latitude    43739 non-null  float64
 6   Drop_Longitude   43739 non-null  float64
 7   Order_Date       43739 non-null  object 
 8   Order_Time       43739 non-null  object 
 9   Pickup_Time      43739 non-null  object 
 10  Weather          43648 non-null  object 
 11  Traffic          43739 non-null  object 
 12  Vehicle          43739 non-null  object 
 13  Area             43739 non-null  object 
 14  Delivery_Time    43739 non-null  int64  
 15  Category         43739 non-null  object 
dtypes: float64(5), int64(2), object(9)
memory usage: 5.3+ MB


In [5]:
df.describe()

,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Delivery_Time
count,43739.000000,43685.000000,43739.000000,43739.000000,43739.000000,43739.000000,43739.000000
mean,29.567137,4.633780,17.210960,70.661177,17.459031,70.821842,124.905645
std,5.815155,0.334716,7.764225,21.475005,7.342950,21.153148,51.915451
min,15.000000,1.000000,-30.902872,-88.366217,0.010000,0.010000,10.000000
25%,25.000000,4.500000,12.933298,73.170283,12.985996,73.280000,90.000000
50%,30.000000,4.700000,18.551440,75.898497,18.633626,76.002574,125.000000
75%,35.000000,4.900000,22.732225,78.045359,22.785049,78.104095,160.000000
max,50.000000,6.000000,30.914057,88.433452,31.054057,88.563452,270.000000


In [6]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


## Data Cleaning

In [7]:
import seaborn as sns

In [8]:
# Set the earth's radius (in kilometers)
R = 6371

# Convert degrees to radians
def deg_to_rad(degrees):
    return degrees * (np.pi/180)

# Function to calculate the distance between two points using the haversine formula
def distcalculate(lat1, lon1, lat2, lon2):
    d_lat = deg_to_rad(lat2-lat1)
    d_lon = deg_to_rad(lon2-lon1)
    a = np.sin(d_lat/2)**2 + np.cos(deg_to_rad(lat1)) * np.cos(deg_to_rad(lat2)) * np.sin(d_lon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c
  
# Calculate the distance between each pair of points
df['Distance'] = np.nan

for i in range(len(df)):
    df.loc[i, 'Distance'] = distcalculate(df.loc[i, 'Store_Latitude'], 
                                        df.loc[i, 'Store_Longitude'], 
                                        df.loc[i, 'Drop_Latitude'], 
                                        df.loc[i, 'Drop_Longitude'])

In [9]:
df['Agent_Age'].unique()

array([37, 34, 23, 38, 32, 22, 33, 35, 36, 21, 24, 29, 25, 31, 27, 26, 20,
       28, 39, 30, 15, 50])

In [10]:
df['Agent_Rating'].unique()

array([4.9, 4.5, 4.4, 4.7, 4.6, 4.8, 4.2, 4.3, 4. , 4.1, 5. , 3.5, 3.8,
       nan, 3.9, 3.7, 2.6, 2.5, 3.6, 3.1, 2.7, 1. , 3.2, 3.3, 6. , 3.4,
       2.8, 2.9, 3. ])

In [11]:
import pandas as pd

# Specify the allowed rating values
allowed_ratings = [4.9, 4.5, 4.4, 4.7, 4.6, 4.8, 4.2, 4.3, 4.0, 4.1, 5.0, 3.5, 3.8, 3.9, 3.7, 2.6, 2.5, 3.6, 3.1, 2.7, 1.0, 3.2, 3.3, 3.4, 2.8, 2.9, 3.0]

# Filter the DataFrame to keep only the allowed rating values
df = df[df['Agent_Rating'].isin(allowed_ratings)]

In [12]:
df['Agent_Rating'].unique()

array([4.9, 4.5, 4.4, 4.7, 4.6, 4.8, 4.2, 4.3, 4. , 4.1, 5. , 3.5, 3.8,
       3.9, 3.7, 2.6, 2.5, 3.6, 3.1, 2.7, 1. , 3.2, 3.3, 3.4, 2.8, 2.9,
       3. ])

In [13]:
print(df['Weather'].unique())
print(df['Area'].unique())
print(df['Traffic'].unique())
print(df['Vehicle'].unique())
print(df['Category'].unique())



['Sunny' 'Stormy' 'Sandstorms' 'Cloudy' 'Fog' 'Windy' nan]
['Urban ' 'Metropolitian ' 'Semi-Urban ' 'Other']
['High ' 'Jam ' 'Low ' 'Medium ' 'NaN ']
['motorcycle ' 'scooter ' 'van' 'bicycle ']
['Clothing' 'Electronics' 'Sports' 'Cosmetics' 'Toys' 'Snacks' 'Shoes'
 'Apparel' 'Jewelry' 'Outdoors' 'Grocery' 'Books' 'Kitchen' 'Home'
 'Pet Supplies' 'Skincare']


In [14]:
import pandas as pd

# Specify the allowed rating values
allowed_traffic = ['High ','Jam ','Low ','Medium ']

df = df[df['Traffic'].isin(allowed_traffic)]

In [15]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category,Distance
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing,3.025149
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics,20.183530
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports,1.552758
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics,7.790401
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys,6.210138


## Trying different models with different features

### Linear Regression

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
X = df[['Agent_Age','Agent_Rating','Distance']]
y=df['Delivery_Time']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
LR = LinearRegression()

In [ ]:
LR.fit(X_train,y_train)

In [ ]:
predictions = LR.predict(X_test)

In [ ]:
print(mean_squared_error(y_test,predictions))

In [ ]:
math.sqrt(2263.365762653255)

In [ ]:
r2 = r2_score(y_test, predictions)
print(f"R-squared: {r2:.2f}")

In [17]:
import math
math.sqrt(2025.2415)

45.00268325333502

In [ ]:
df.head()

## Mapping to enhance model and checking the linear accuracy

In [18]:
print(df['Weather'].unique())
print(df['Area'].unique())
print(df['Traffic'].unique())
print(df['Vehicle'].unique())
print(df['Category'].unique())


['Sunny' 'Stormy' 'Sandstorms' 'Cloudy' 'Fog' 'Windy']
['Urban ' 'Metropolitian ' 'Semi-Urban ' 'Other']
['High ' 'Jam ' 'Low ' 'Medium ']
['motorcycle ' 'scooter ' 'van']
['Clothing' 'Electronics' 'Sports' 'Cosmetics' 'Toys' 'Snacks' 'Shoes'
 'Apparel' 'Jewelry' 'Outdoors' 'Grocery' 'Books' 'Kitchen' 'Home'
 'Pet Supplies' 'Skincare']


In [20]:
weather_unique = df['Weather'].unique()

# Create a mapping dictionary to replace the unique values
weather_mapping = {weather: i for i, weather in enumerate(weather_unique)}

# Replace the unique values with the mapped values
df['Weather_encoded'] = df['Weather'].map(weather_mapping)

# Now you can use the encoded weather column as a feature in your linear regression model
X = df[['Weather_encoded']]
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(math.sqrt(mean_squared_error(y_test,predictions)))

50.75275823292387


In [21]:
area_unique = df['Area'].unique()

# Create a mapping dictionary to replace the unique values
area_mapping = {area: i for i, area in enumerate(area_unique)}

# Replace the unique values with the mapped values
df['Area_encoded'] = df['Area'].map(area_mapping)

# Now you can use the encoded area column as a feature in your linear regression model
X = df[['Area_encoded']]
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(math.sqrt(mean_squared_error(y_test,predictions)))

50.989281511961615


In [22]:
traffic_unique = df['Traffic'].unique()

# Create a mapping dictionary to replace the unique values
traffic_mapping = {traffic: i for i, traffic in enumerate(traffic_unique)}

# Replace the unique values with the mapped values
df['Traffic_encoded'] = df['Traffic'].map(traffic_mapping)

# Now you can use the encoded traffic column as a feature in your linear regression model
X = df[['Traffic_encoded']]
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(math.sqrt(mean_squared_error(y_test,predictions)))

50.7338528981299


In [23]:
vehicle_unique = df['Vehicle'].unique()

# Create a mapping dictionary to replace the unique values
vehicle_mapping = {vehicle: i for i, vehicle in enumerate(vehicle_unique)}

# Replace the unique values with the mapped values
df['Vehicle_encoded'] = df['Vehicle'].map(vehicle_mapping)

# Now you can use the encoded traffic column as a feature in your linear regression model
X = df[['Vehicle_encoded']]
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(math.sqrt(mean_squared_error(y_test,predictions)))

50.805125843238606


In [24]:
category_unique = df['Category'].unique()

# Create a mapping dictionary to replace the unique values
category_mapping = {category: i for i, category in enumerate(category_unique)}

# Replace the unique values with the mapped values
df['Category_encoded'] = df['Category'].map(category_mapping)

# Now you can use the encoded category column as a feature in your linear regression model
X = df[['Category_encoded']]
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(math.sqrt(mean_squared_error(y_test,predictions)))

51.21457897841234


In [25]:
df.sample(10)

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,...,Vehicle,Area,Delivery_Time,Category,Distance,Weather_encoded,Area_encoded,Traffic_encoded,Vehicle_encoded,Category_encoded
11103,pdgk880799433,24,4.9,13.081878,80.248519,13.091878,80.258519,2022-04-01,11:30:00,11:40:00,...,motorcycle,Metropolitian,125,Skincare,1.552247,3,1,0,0,15
42749,wgyl529162796,35,5.0,13.049645,80.242268,13.059645,80.252268,2022-03-13,11:00:00,11:10:00,...,motorcycle,Metropolitian,145,Pet Supplies,1.552346,0,1,2,0,14
38584,hxjp975012600,31,4.8,26.888420,75.800689,26.908420,75.820689,2022-03-28,10:15:00,10:20:00,...,motorcycle,Metropolitian,15,Grocery,2.979796,3,1,2,0,10
2861,rlrn877316570,24,5.0,19.888716,75.321461,19.938716,75.371461,2022-02-11,23:30:00,23:45:00,...,scooter,Other,50,Outdoors,7.631222,2,3,2,1,9
22456,vzns916314737,27,4.7,17.450851,78.379347,17.530851,78.459347,2022-03-27,23:00:00,23:05:00,...,motorcycle,Metropolitian,80,Clothing,12.292885,2,1,2,0,0
22007,yaud276334044,34,3.9,12.337928,76.617889,12.407928,76.687889,2022-03-06,18:10:00,18:15:00,...,motorcycle,Metropolitian,155,Snacks,10.880652,2,1,3,0,5
42295,gjec644150783,31,4.8,19.126630,72.829976,19.216630,72.919976,2022-03-18,23:35:00,23:45:00,...,motorcycle,Urban,85,Apparel,13.765935,2,0,2,0,7
5560,vhjm046034832,38,5.0,0.000000,0.000000,0.020000,0.020000,2022-04-05,09:10:00,09:15:00,...,motorcycle,Urban,85,Sports,3.145067,3,0,2,0,2
19447,syec951249665,30,5.0,22.310237,73.158921,22.380237,73.228921,2022-03-06,23:20:00,23:25:00,...,motorcycle,Metropolitian,130,Snacks,10.602507,1,1,2,0,5
5981,cxdq919532708,22,4.9,11.026117,76.944652,11.106117,77.024652,2022-04-02,22:35:00,22:50:00,...,scooter,Urban,65,Electronics,12.463861,1,0,2,1,1


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43594 entries, 0 to 43738
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          43594 non-null  object 
 1   Agent_Age         43594 non-null  int64  
 2   Agent_Rating      43594 non-null  float64
 3   Store_Latitude    43594 non-null  float64
 4   Store_Longitude   43594 non-null  float64
 5   Drop_Latitude     43594 non-null  float64
 6   Drop_Longitude    43594 non-null  float64
 7   Order_Date        43594 non-null  object 
 8   Order_Time        43594 non-null  object 
 9   Pickup_Time       43594 non-null  object 
 10  Weather           43594 non-null  object 
 11  Traffic           43594 non-null  object 
 12  Vehicle           43594 non-null  object 
 13  Area              43594 non-null  object 
 14  Delivery_Time     43594 non-null  int64  
 15  Category          43594 non-null  object 
 16  Distance          43594 non-null  float64
 17

## Some more trial models

In [ ]:
X = df[['Agent_Age','Agent_Rating','Distance','Weather_encoded','Area_encoded','Traffic_encoded','Vehicle_encoded','Category_encoded']]
y = df['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print('RMSE: ',math.sqrt(mean_squared_error(y_test,predictions)))
print('Accuracy:',100-math.sqrt(mean_squared_error(y_test,predictions)))

In [ ]:
from sklearn.preprocessing import StandardScaler

# Feature variables
X = df[['Agent_Age', 'Agent_Rating', 'Distance', 'Weather_encoded', 'Area_encoded', 'Traffic_encoded', 'Vehicle_encoded', 'Category_encoded']]

# Target variable
y = df['Delivery_Time']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the feature variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions on the test set
predictions = model.predict(X_test_scaled)

# Evaluate the model
rmse = math.sqrt(mean_squared_error(y_test, predictions))
accuracy = 100 - rmse

print('RMSE: ', rmse)
print('Accuracy: ', accuracy)



In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Feature variables
X = df[['Agent_Age', 'Agent_Rating', 'Distance', 'Weather_encoded', 'Area_encoded', 'Traffic_encoded', 'Vehicle_encoded', 'Category_encoded']]

# Target variable
y = df['Delivery_Time']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize the feature variables
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the linear regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions on the test set
predictions = model.predict(X_test_scaled)

# Evaluate the model
rmse = math.sqrt(mean_squared_error(y_test, predictions))
accuracy = 100 - rmse

print('RMSE: ', rmse)
print('Accuracy: ', accuracy)

### Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
accuracy = 100 - rmse

print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%")

### Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
accuracy = 100 - rmse

print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%")

### Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
accuracy = 100 - rmse

print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%")

### Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
accuracy = 100 - rmse

print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%")

### SVR 

In [ ]:
from sklearn.svm import SVR

model = SVR(kernel='rbf')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
accuracy = 100 - rmse

print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%")

### K Neighbours Regressor

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=5)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
accuracy = 100 - rmse

print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%")

## Final Model

In [27]:
X = df[['Agent_Age', 'Agent_Rating', 'Distance', 'Weather_encoded', 'Area_encoded', 'Traffic_encoded', 'Vehicle_encoded', 'Category_encoded']]

# Target variable
y = df['Delivery_Time']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
from sklearn.ensemble import RandomForestRegressor

rfmodel = RandomForestRegressor(n_estimators=100, random_state=42)
rfmodel.fit(X_train, y_train)

y_pred = rfmodel.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mse)
accuracy = 100 - rmse

print(f"RMSE: {rmse:.2f}")
print(f"Accuracy: {accuracy:.2f}%")

RMSE: 23.28
Accuracy: 76.72%


In [29]:
features = np.array([30,4.2,17.23,1,1,1,2,7])

In [31]:
rfmodel.predict([features])

array([178.8])

In [32]:
import pickle

# Save the model to a pickle file
with open('rfdelivery_model.pkl', 'wb') as file:
    pickle.dump(rfmodel, file)

In [ ]:
import joblib
joblib.dump(model, 'model.joblib')